# Local Training

Fast local validation of training configs (focal loss, LoRA, etc.) using
real MERMAID data from S3 with the real DINOv3 encoder. Completes in
under 5 minutes by limiting iterations per epoch.

**Requirements:**
- AWS credentials (`AWS_PROFILE=mermaid-core`) for S3 data access
- HuggingFace cache for DINOv3 encoder (~350 MB one-time download)
- No MLflow — metrics returned as a dict

In [ ]:
%load_ext autoreload
%autoreload 2
%autosave 30

In [ ]:
import os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
import time

import matplotlib.pyplot as plt
import torch
from torch.utils.data import ConcatDataset, DataLoader

from mermaidseg.dataset_reconciliation import (
    SourceLabelRegistry,
    attach_registry,
    prepare_splits_for_registry,
)
from mermaidseg.datasets import MermaidDataset, worker_init_fn
from mermaidseg.io import get_parser, setup_config, update_config_with_args
from mermaidseg.model.eval import Evaluator
from mermaidseg.model.meta import MetaModel
from mermaidseg.model.train import train_model

## 1. Device

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    for i in range(torch.cuda.device_count()):
        print(f"CUDA Device {i}: {torch.cuda.get_device_name(i)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Using CPU — expect ~4 min for default settings")

seed = 42
torch.manual_seed(seed)
print(f"Device: {device}")

## 2. Config

Loads the focal loss training config with small iteration overrides.
Edit `args_input` to test different configs or parameters.

In [ ]:
config_path_dict = {
    "data": "../configs/data_config_local.yaml",
    "model": "../configs/model_config.yaml",
    "training": "../configs/training_config_dinov3_focal.yaml",
    "logger": "../configs/logger_config.yaml",
}

cfg = setup_config(config_path_dict)

args_input = [
    "--run-name=local-focal-test",
    "--epochs=3",
    "--iterations-per-train-epoch=10",
    "--iterations-per-val-epoch=5",
    "--batch-size=4",
]

parser = get_parser()
args = parser.parse_args(args_input)
cfg = update_config_with_args(cfg, args)

print(f"Run name: {cfg.run_name}")
print(f"Loss: {cfg.training.loss.type}")
print(f"Epochs: {cfg.training.epochs}")
print(f"Train iters/epoch: {cfg.training.iterations_per_train_epoch}")
print(f"Val iters/epoch: {cfg.training.iterations_per_val_epoch}")
print(f"Batch size: {cfg.training.batch_size}")
print(f"Encoder frozen: {cfg.training.freeze_encoder}")

## 3. Data

In [ ]:
num_workers = 0

dataset_dict = {}
dataset_cls_mapping = {"mermaid": MermaidDataset}

for dataset_name in cfg.data:
    if dataset_name not in dataset_cls_mapping:
        continue
    dataset_cls = dataset_cls_mapping[dataset_name]
    for split in cfg.data[dataset_name]:
        if cfg.data[dataset_name][split] is None or cfg.data[dataset_name][split] == "None":
            continue
        dataset_kwargs = dict(cfg.data[dataset_name][split])
        dataset_kwargs.setdefault("padding", cfg.training.padding)
        dataset_dict[dataset_name, split] = dataset_cls(**dataset_kwargs)

for (name, split), ds in dataset_dict.items():
    print(f"{name} - {split}: {len(ds)} samples")

In [ ]:
_, registry_datasets = prepare_splits_for_registry(dataset_dict)

registry = SourceLabelRegistry(
    registry_datasets,
    target_label_subset=cfg.training.class_subset,
    compute_concepts=False,
    concept_mapping_path=None,
    label_roll_up=cfg.training.get("label_roll_up", False),
).to(device)

attach_registry(registry, dataset_dict.values())

print(f"Target classes: {registry.num_target_classes}")
print(f"source_to_target shape: {registry.source_to_target.shape}")

In [ ]:
from mermaidseg.datasets.base_dataset import BaseCoralDataset

train_loader = DataLoader(
    ConcatDataset([ds for (name, split), ds in dataset_dict.items() if split == "train"]),
    batch_size=cfg.training.batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    drop_last=True,
    collate_fn=BaseCoralDataset.collate_fn,
    worker_init_fn=worker_init_fn,
)

val_loader = DataLoader(
    ConcatDataset([ds for (name, split), ds in dataset_dict.items() if split == "val"]),
    batch_size=cfg.training.batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    drop_last=True,
    collate_fn=BaseCoralDataset.collate_fn,
    worker_init_fn=worker_init_fn,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 4. Model

In [ ]:
meta_model = MetaModel(
    run_name=cfg.run_name,
    num_classes=registry.num_target_classes,
    num_concepts=None,
    device=device,
    model_kwargs=cfg.model.copy(),
    training_kwargs=cfg.training.copy(),
    source_to_target_lookup=registry.source_to_target,
)

evaluator = Evaluator(
    num_classes=registry.num_target_classes,
    device=device,
)

total_params = sum(p.numel() for p in meta_model.model.parameters())
trainable_params = sum(p.numel() for p in meta_model.model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Frozen params: {total_params - trainable_params:,}")
print(f"Loss: {meta_model.loss.__class__.__name__}")

## 5. Train

In [ ]:
t0 = time.time()

metrics = train_model(
    meta_model,
    evaluator,
    train_loader,
    val_loader,
    logger=None,
)

elapsed = time.time() - t0
print(f"\nTraining completed in {elapsed:.1f}s ({elapsed / 60:.1f} min)")

## 6. Results

In [ ]:
epochs = sorted(metrics.keys())
train_loss = [metrics[e]["loss"]["train/loss"] for e in epochs]
val_loss = [metrics[e]["loss"]["validation/loss"] for e in epochs]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_loss, "o-", label="train")
axes[0].plot(epochs, val_loss, "o-", label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Focal Loss")
axes[0].legend()

if "train_metrics" in metrics[epochs[0]]:
    train_miou = [metrics[e]["train_metrics"].get("miou", float("nan")) for e in epochs]
    val_miou = [metrics[e].get("validation_metrics", {}).get("miou", float("nan")) for e in epochs]
    axes[1].plot(epochs, train_miou, "o-", label="train")
    axes[1].plot(epochs, val_miou, "o-", label="val")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("mIoU")
    axes[1].set_title("Mean IoU")
    axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
last = metrics[epochs[-1]]
print("Final epoch metrics:")
print(f"  Train loss: {last['loss']['train/loss']:.4f}")
print(f"  Val loss:   {last['loss']['validation/loss']:.4f}")
if "train_metrics" in last:
    for name, val in last["train_metrics"].items():
        if not isinstance(val, (int, float)):
            continue
        print(f"  Train {name}: {val:.4f}")
if "validation_metrics" in last:
    for name, val in last["validation_metrics"].items():
        if not isinstance(val, (int, float)):
            continue
        print(f"  Val {name}: {val:.4f}")